In [ ]:
import yfinance as yf
import pandas as pd
import datetime
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
ticker= '^GSPC'
start = datetime.datetime(2007, 1, 1)
end = datetime.datetime(2025, 12, 31)
s_p500 = yf.download(ticker, start=start, end = end)

In [ ]:
ret = 100*s_p500.Close.pct_change().dropna()
ret_vol = ret.rolling(5).std()

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(ret_vol.index,ret_vol)
plt.title('S&P-500 Return Volatility')
plt.ylabel('Volatility')
plt.show()

In [ ]:
pip install arch

from arch import arch_model

arch = arch_model(ret, mean='zero', vol='ARCH', p=1).fit(disp='off')
arch.summary()

In [ ]:
bic_arch = []

prange = range(1,5)

for p in prange:
    arch = arch_model(ret, mean='zero', vol='ARCH', p=p).fit(disp='off')
    bic_arch.append(arch.bic)

best_param = prange[np.argmin(bic_arch)]
arch = arch_model(ret, mean='zero', vol='ARCH', p=best_param).fit(disp='off')
arch.summary()

In [ ]:
split_date = ret.iloc[-200:].index
forecast = arch.forecast(start=split_date[0])
forecast_arch = forecast

In [ ]:
from sklearn.metrics import mean_squared_error as mse

rmse_arch = np.sqrt(mse(ret_vol[-200:] / 100,np.sqrt(forecast_arch.variance.iloc[-len(split_date):]/ 100)))
rmse_arch

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(ret_vol / 100, label='Return Volatility')
plt.plot(forecast_arch.variance.iloc[-len(split_date):] / 100,label='Predicted Volatility')
plt.title('Volatility Prediction with ARCH', fontsize=12)
plt.legend()
plt.show()

In [ ]:
ret_vol = ret.rolling(5).std()
ret_vol = pd.DataFrame(ret_vol)
ret_vol.reset_index(drop=True, inplace=True)

In [ ]:
returns_svm = ret ** 2
returns_svm = returns_svm.reset_index()
returns_svm = returns_svm.drop("Date",axis=1)

In [ ]:
X = pd.concat([ret_vol, returns_svm], axis=1, ignore_index=True)
X = X[4:].copy()
X = X.reset_index()
X.drop('index', axis=1, inplace=True)

In [ ]:
ret_vol = ret_vol.dropna().reset_index()
ret_vol.drop('index', axis=1, inplace=True)

In [ ]:
from sklearn.svm import SVR
svr_lin = SVR(kernel='linear')
svr_rbf = SVR(kernel='rbf')

In [ ]:
from scipy.stats import uniform as sp_rand
from sklearn.model_selection import RandomizedSearchCV
n = 200
para_grid = {'gamma': sp_rand(),'C': sp_rand(),'epsilon': sp_rand()}
clf = RandomizedSearchCV(svr_lin, para_grid)
clf.fit(X.iloc[:-n].values,ret_vol.iloc[1:-(n-1)].values.reshape(-1,))
predict_svr_lin = clf.predict(X.iloc[-n:])

In [ ]:
predict_svr_lin = pd.DataFrame(predict_svr_lin)
predict_svr_lin.index = ret.iloc[-n:].index

In [ ]:
rmse_svr = np.sqrt(mse(ret_vol.iloc[-n:] / 100,predict_svr_lin / 100))
ret_vol.index = ret.iloc[4:].index

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(ret_vol / 100, label='Return Volatility')
plt.plot(predict_svr_lin / 100, label='Predicted Volatility')
plt.title('SVR-GARCH (Linear), RMSE = {:.6f}'.format(rmse_svr))
plt.legend()
plt.show()

In [ ]:
para_grid ={'gamma': sp_rand(),'C': sp_rand(),'epsilon': sp_rand()}
clf = RandomizedSearchCV(svr_rbf, para_grid)
clf.fit(X.iloc[:-n].values,ret_vol.iloc[1:-(n-1)].values.reshape(-1,))
predict_svr_rbf = clf.predict(X.iloc[-n:])

In [ ]:
predict_svr_rbf = pd.DataFrame(predict_svr_rbf)
predict_svr_rbf.index = ret.iloc[-n:].index
rmse_svr_rbf = np.sqrt(mse(ret_vol.iloc[-n:] / 100,predict_svr_rbf / 100))

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(ret_vol / 100, label='Return Volatility')
plt.plot(predict_svr_rbf / 100, label='Predicted Volatility')
plt.title('SVR-GARCH (RBF), RMSE = {:.6f}'.format(rmse_svr_rbf))
plt.legend()
plt.show()